In [3]:
using DataFrames, DifferentialEquations, Plots, CSV, Random, Lux, Optimization, OptimizationOptimisers, Zygote
using ComponentArrays, SciMLSensitivity, JLD2, StaticArrays
data_loaded = CSV.read("C:/Ude_HH_model/ude_models/synthetic_data.csv", DataFrame)
# all constants
true_V = Float32.(data_loaded[!, "V"])
t_steps = Float32.(data_loaded[!, "Time"])
# all constants
g_na = 120.0f0
g_k = 36.0f0
g_l = 0.3f0
c_m = 1.0f0
I_ext = 10.0f0
E_na = 50.0f0
E_k = -77.0f0
E_l = -54.4f0
 

-54.4f0

In [4]:
# rate_functions
# rate funtions
typeof(c_m)
# --- Potassium Gating (n) ---
alpha_n(V) = abs(V + 55.0f0) < 1.0f-6 ? 0.1f0 : 0.01f0 * (V + 55.0f0) / (1.0f0 - exp(-(V + 55.0f0) / 10.0f0))
beta_n(V) = 0.125f0 * exp(-(V + 65.0f0) / 80.0f0)

beta_n (generic function with 1 method)

In [5]:
# --- Sodium Activation (m) ---
alpha_m(V) = abs(V + 40.0f0) < 1.0f-6 ? 1.0f0 : 0.1f0 * (V + 40.0f0) / (1.0f0 - exp(-(V + 40.0f0) / 10.0f0))
beta_m(V) = 4.0f0 * exp(-(V + 65.0f0) / 18.0f0)

beta_m (generic function with 1 method)

In [6]:
# --- Sodium Inactivation (h) ---
alpha_h(V) = 0.07f0 * exp(-(V + 65.0f0) / 20.0f0)
beta_h(V) = 1.0f0 / (1.0f0 + exp(-(V + 35.0f0) / 10.0f0))

beta_h (generic function with 1 method)

In [7]:
rng = Random.seed!(42)

TaskLocalRNG()

In [12]:
nn = Chain(Dense(1 => 32),
    Dense(32 => 32, tanh),
    Dense(32 => 1, sigmoid))
ps, st = Lux.setup(rng, nn)

((layer_1 = (weight = Float32[-0.15767582; -1.1307708; … ; 0.19089252; 0.42486224;;], bias = Float32[0.6402823, -0.15548313, 0.3585751, 0.47565484, -0.83354366, -0.7983309, -0.9632733, -0.35786855, 0.1624806, 0.93903005  …  0.86745214, 0.8148211, 0.38635254, -0.62182, -0.18979192, -0.6013012, -0.8030517, 0.56790996, -0.28980362, 0.96977556]), layer_2 = (weight = Float32[0.37709013 -0.4490932 … -0.3449148 -0.36564463; 0.054253716 0.49733526 … 0.35948935 0.24132304; … ; 0.49617618 0.08238792 … 0.18305631 -0.1491749; -0.021193195 -0.16762535 … -0.43130484 0.3783914], bias = Float32[0.03878094, -0.007602322, -0.11970364, -0.098564304, -0.025683658, -0.039688274, -0.12524073, 0.012478712, -0.054133095, 0.12872824  …  0.040868536, 0.09221246, -0.08388274, 0.08786137, -0.1030199, -0.08795758, 0.07973033, 0.059021942, 0.066383034, 0.067760795]), layer_3 = (weight = Float32[-0.29468712 -0.103986636 … -0.19075362 -0.13402101], bias = Float32[0.10804534])), (layer_1 = NamedTuple(), layer_2 = Name

In [8]:
function ude_hh!(du, u_0, p, t)

    V, m, h, n = u_0
    ps = p

    pred_n, _ = nn(@SVector[n], ps, st)






    du[1] = 1 / c_m * (I_ext - g_na * m^3 * h * (V - E_na) - g_k * pred_n[1] * (V - E_k) - g_l * (V - E_l))
    du[2] = alpha_m(V) * (1 - m) - beta_m(V) * (m)
    du[3] = alpha_h(V) * (1 - h) - beta_h(V) * (h)
    du[4] = alpha_n(V) * (1 - n) - beta_n(V) * (n)
end

ude_hh! (generic function with 1 method)

In [13]:
u_0 = [-65.0f0, 0.05f0, 0.6f0, 0.317f0]

tspan = (0.0f0, 30.0f0)
p = [g_na, g_k, g_l, c_m, I_ext, E_na, E_k, E_l, ps, st]
prob = ODEProblem(ude_hh!, u_0, tspan, p)


┌ Warning: Using arrays or dicts to store parameters of different types can hurt performance.
│ Consider using tuples instead.
└ @ SciMLBase C:\Users\ADMIN\.julia\packages\SciMLBase\O1HPI\src\performance_warnings.jl:32


ODEProblem with uType Vector{Float32} and tType Float32. In-place: true
Non-trivial mass matrix: false
timespan: (0.0f0, 30.0f0)
u0: 4-element Vector{Float32}:
 -65.0
   0.05
   0.6
   0.317

In [15]:
function loss_function(ps, p)


    prob = ODEProblem(ude_hh!, u_0, tspan, ps)
    sol = solve(prob, Tsit5(), reltol=1e-6, abstol=1e-6, saveat=t_steps, sensealg=InterpolatingAdjoint())
    pred_V = sol[1, :]
    loss = sum(abs2, pred_V - true_V) / length(true_V)
    return loss
end

loss_function (generic function with 1 method)

Optimization

In [16]:
function callback(state, l)
    if state.iter % 5 == 0
        println("current iteration = $(state.iter)  | current_loss = $(l)")
        if state.iter % 250 == 0
            jldsave("C:/Ude_HH_model/ude_models/leaning_parameter.jld2"; p=state.u)
        end
    end
    return false
end

callback (generic function with 1 method)

In [17]:
opt = OptimizationFunction(loss_function, AutoZygote())

OptimizationFunction{true, AutoZygote, typeof(loss_function), Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, typeof(SciMLBase.DEFAULT_OBSERVED_NO_TIME), Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing}(Main.loss_function, AutoZygote(), nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, SciMLBase.DEFAULT_OBSERVED_NO_TIME, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing)

In [18]:
# Optimization
println(typeof(ps))
net = ComponentArray(ps)
opt_prob = OptimizationProblem(opt, net)

@NamedTuple{layer_1::@NamedTuple{weight::Matrix{Float32}, bias::Vector{Float32}}, layer_2::@NamedTuple{weight::Matrix{Float32}, bias::Vector{Float32}}, layer_3::@NamedTuple{weight::Matrix{Float32}, bias::Vector{Float32}}}


OptimizationProblem. In-place: true
u0: ComponentVector{Float32}(layer_1 = (weight = Float32[-0.15767582; -1.1307708; … ; 0.19089252; 0.42486224;;], bias = Float32[0.6402823, -0.15548313, 0.3585751, 0.47565484, -0.83354366, -0.7983309, -0.9632733, -0.35786855, 0.1624806, 0.93903005  …  0.86745214, 0.8148211, 0.38635254, -0.62182, -0.18979192, -0.6013012, -0.8030517, 0.56790996, -0.28980362, 0.96977556]), layer_2 = (weight = Float32[0.37709013 -0.4490932 … -0.3449148 -0.36564463; 0.054253716 0.49733526 … 0.35948935 0.24132304; … ; 0.49617618 0.08238792 … 0.18305631 -0.1491749; -0.021193195 -0.16762535 … -0.43130484 0.3783914], bias = Float32[0.03878094, -0.007602322, -0.11970364, -0.098564304, -0.025683658, -0.039688274, -0.12524073, 0.012478712, -0.054133095, 0.12872824  …  0.040868536, 0.09221246, -0.08388274, 0.08786137, -0.1030199, -0.08795758, 0.07973033, 0.059021942, 0.066383034, 0.067760795]), layer_3 = (weight = Float32[-0.29468712 -0.103986636 … -0.19075362 -0.13402101], bias =

In [ ]:
println("Lets start the trainig ====================================== ")
solve(opt_prob, Adam(0.05), callback=callback, maxiters=10)  

--------------------------------------------